# Tarea 3

In [15]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, TwoSlopeNorm

# Direcciones de los directorios de datos en el repositorio del curso
TAREA3 = "https://github.com/nestorveas/Tarea3_gf0657"

In [25]:
asadas = gpd.read_file("Entes.shp")
cantones = gpd.read_file("cantones-2020-simplificados.gpkg")

In [33]:
cantones_cr05 = cantones.to_crs(5367)

In [18]:
asadas.shape

(1372, 44)

In [31]:
asadas.crs

<Projected CRS: EPSG:5367>
Name: CR05 / CRTM05
Axis Info [cartesian]:
- N[north]: Northing (metre)
- E[east]: Easting (metre)
Area of Use:
- name: Costa Rica - onshore and offshore east of 86°30'W.
- bounds: (-86.5, 2.21, -81.43, 11.77)
Coordinate Operation:
- name: Costa Rica TM 2005
- method: Transverse Mercator
Datum: Costa Rica 2005
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [20]:
asadas.dtypes

Región             str
Ente_Opera         str
PROVINCIA          str
Cantón             str
DISTRITO           str
F______DAT      object
Teléfono_          str
Fax                str
Correo             str
Dirección          str
F______COO      object
Este           float64
Norte          float64
F___COORDE      object
Longitud       float64
Latitud        float64
F______CAR      object
Local_Aten         str
Tipo_Admin         str
Tipo_Ubica         str
Calificaci         str
Dominio            str
HasthTAG           str
F______SER      object
Fijo_Domic       int64
Medido_Dom       int64
Fijo_Empre       int64
Medido_Emp       int64
Hidrantes        int64
Servicios_       int64
Poblacion_       int64
F______S_1      object
Serv_Proy_       int64
Pobl_Abast       int64
F______COM      object
IDEO               str
PKENTEOPER       int64
Comunidade         str
Cantidad_F       int64
Total_Caud       int64
Caudal_Req       int64
Aprobados_       int64
URL_IMAGEN      object
geometry   

In [26]:
cantones.head()

,gmlid,objectid,cod_catalo,cod_canton,canton,ori_toponi,area,cod_provin,provincia,version,geometry
0,limitecantonal_5k.1,1.0,160104,610,Corredores,Tiene su origen en el topónimo del río Corredo...,623.61,6,Puntarenas,20200429,"POLYGON ((-82.9356 8.42366, -82.95353 8.41391,..."
1,limitecantonal_5k.2,2.0,160104,607,Golfito,"Se debe a la forma que presenta el litoral, d...",1753.42,6,Puntarenas,20200429,"MULTIPOLYGON (((-83.46678 8.68886, -83.46655 8..."
2,limitecantonal_5k.3,3.0,160104,608,Coto Brus,De origen indígena las palabras Coto (couto) B...,944.24,6,Puntarenas,20200429,"POLYGON ((-82.90501 8.77425, -82.90654 8.77489..."
3,limitecantonal_5k.4,4.0,160104,605,Osa,"En recuerdo del cacique Osa, que en los inicio...",1932.03,6,Puntarenas,20200429,"POLYGON ((-83.83977 9.25543, -83.83642 9.25533..."
4,limitecantonal_5k.5,5.0,160104,603,Buenos Aires,El nombre se le dio por la brisa constante que...,2382.94,6,Puntarenas,20200429,"POLYGON ((-83.32144 9.38418, -83.31946 9.38414..."


In [28]:
asadas_limpio = asadas[["Ente_Opera", "DISTRITO", "IDEO", "Este", "Norte", "Tipo_Admin", "Calificaci", "Servicios_", "Poblacion_", "geometry"]]
asadas_limpio.head()

,Ente_Opera,DISTRITO,IDEO,Este,Norte,Tipo_Admin,Calificaci,Servicios_,Poblacion_,geometry
0,"00719-2014_EL HUMO DE PEJIBAYE DE JIMENEZ, CAR...",PEJIBAYE,00719-2014,531457.00,1086443.00,ASADA,C,0,0,POINT (531457 1086443)
1,"00015-2014_SAN JOSE (PIZOTE) DE UPALA, ALAJUELA",SAN JOSE O PIZOTE,00015-2014,375817.87,1212246.91,ASADA,A,2433,8515,POINT (375817.87 1212246.91)
2,"00023-2014_EL PROGRESO DE SAN JOSE DE UPALA, A...",SAN JOSE O PIZOTE,00023-2014,364679.07,1215110.70,ASADA,B,165,577,POINT (364679.07 1215110.7)
3,00029-2014_SAN MIGUEL DE PIEDADES SUR DE SAN R...,PIEDADES SUR,00029-2014,443344.28,1118805.74,ASADA,B,235,822,POINT (443344.28 1118805.74)
4,"00334-2014_SAN JOSE DEL HIGUERON, LA ANGOSTURA...",SANTIAGO,00334-2014,436037.82,1108432.77,ASADA,B,107,374,POINT (436037.82 1108432.77)


In [37]:
#Uso de la funcion sjoin
#En que cantón está cada asada

asadas_canton = gpd.sjoin(
    asadas_limpio,
    cantones_cr05[["cod_canton", "canton", "provincia", "geometry"]], #Solo esas lineas porque no ocupo todas, si me traigo la geometria porque es importante para la forma del shape
    how="left",
    predicate="within" 
)

# Asadas por cantón, de vuelta en la capa de países
num_asadas_canton = asadas_canton.groupby("canton").size().rename("asadas").sort_values(ascending=False)

num_asadas_canton
#asadas_canton 
#print(aeropuertos_pais.shape)
#aeropuertos_pais.head()

canton
San Carlos             77
Pérez Zeledón          72
Nicoya                 69
Turrialba              62
Puntarenas             59
                       ..
Goicoechea              1
Montes de Oca           1
Escazú                  1
Santo Domingo           1
Vázquez de Coronado     1
Name: asadas, Length: 75, dtype: int64

In [39]:
# Asadas por provincia, para el gráfico
num_asadas_provincia = asadas_canton.groupby("provincia").size().rename("asadas").sort_values(ascending=False)
num_asadas_provincia

provincia
Alajuela      321
Guanacaste    281
San José      263
Puntarenas    207
Limón         130
Cartago       129
Heredia        41
Name: asadas, dtype: int64

In [23]:
asadas_canton =asadas_limpio.groupby("Cantón")[["Servicios_", "Poblacion_"]].sum()

KeyError: 'Cantón'

In [ ]:
asadas_canton["num_asadas"] = asadas_limpio.groupby("Cantón").size()
asadas_canton.head()

,Servicios_,Poblacion_,num_asadas
Cantón,,,
ABANGARES,3157,11044,24
ACOSTA,4549,15916,28
ALAJUELA,17770,62189,38
ALVARADO,315,1101,3
ASERRI,8604,30109,27


In [ ]:
cantones = gpd.read_file("Tarea3/cantones-2020-simplificados.gpkg")


In [ ]:
type(cantones)

geopandas.geodataframe.GeoDataFrame

In [ ]:
cantones.shape

(82, 11)

In [ ]:
cantones.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [ ]:
cantones.dtypes

gmlid              str
objectid       float64
cod_catalo         str
cod_canton       int32
canton             str
ori_toponi         str
area           float64
cod_provin       int32
provincia          str
version            str
geometry      geometry
dtype: object

In [ ]:
cantones_cr05 = cantones.to_crs(5367)

In [ ]:
cantones_cr05.head()

,gmlid,objectid,cod_catalo,cod_canton,canton,ori_toponi,area,cod_provin,provincia,version,geometry
0,limitecantonal_5k.1,1.0,160104,610,Corredores,Tiene su origen en el topónimo del río Corredo...,623.61,6,Puntarenas,20200429,"POLYGON ((617212.927 931573.126, 615240.87 930..."
1,limitecantonal_5k.2,2.0,160104,607,Golfito,"Se debe a la forma que presenta el litoral, d...",1753.42,6,Puntarenas,20200429,"MULTIPOLYGON (((558675.69 960783.21, 558700.6 ..."
2,limitecantonal_5k.3,3.0,160104,608,Coto Brus,De origen indígena las palabras Coto (couto) B...,944.24,6,Puntarenas,20200429,"POLYGON ((620471.4 970360.4, 620302.44 970430...."
3,limitecantonal_5k.4,4.0,160104,605,Osa,"En recuerdo del cacique Osa, que en los inicio...",1932.03,6,Puntarenas,20200429,"POLYGON ((517604.014 1023402.771, 517972.111 1..."
4,limitecantonal_5k.5,5.0,160104,603,Buenos Aires,El nombre se le dio por la brisa constante que...,2382.94,6,Puntarenas,20200429,"POLYGON ((574526.584 1037709.604, 574744.74 10..."
